<a href="https://colab.research.google.com/github/TediBalint/AI-Jegyzetek/blob/master/Pseudo%20Labeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pseudo-Labeling

Ebben a notebookban a **pseudo-labeling** (álcímkézés) módszerét vizsgáljuk.

## Tartalomjegyzék

1. Semi-supervised learning alapjai
2. Pseudo-labeling algoritmus
3. Confidence threshold
4. Self-training
5. Gyakorlati alkalmazás

## 1. Semi-supervised learning alapjai

### Probléma

| Típus | Címkézett | Címkézetlen | Költség |
|-------|-----------|-------------|--------|
| Supervised | Sok | Nincs | Drága |
| Unsupervised | Nincs | Sok | Olcsó |
| **Semi-supervised** | Kevés | Sok | Hatékony |

### Alapfeltevések

1. **Smoothness**: Közeli pontok → hasonló címke
2. **Cluster**: Azonos klaszter → azonos címke
3. **Manifold**: Adatok alacsony dimenziós sokaságon

### Pseudo-labeling ötlete

Címkézetlen adatokra a modell predikciója lesz a "címke".

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.semi_supervised import SelfTrainingClassifier
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# Adat generálás
X, y = make_moons(n_samples=500, noise=0.2, random_state=42)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Csak kevés címkézett adat
n_labeled = 20  # Csak 20 címkézett példa
labeled_idx = np.random.choice(len(X_train), n_labeled, replace=False)
unlabeled_idx = np.setdiff1d(np.arange(len(X_train)), labeled_idx)

X_labeled = X_train[labeled_idx]
y_labeled = y_train[labeled_idx]
X_unlabeled = X_train[unlabeled_idx]
y_unlabeled_true = y_train[unlabeled_idx]  # Csak kiértékeléshez

# Vizualizáció
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Teljes adat
axes[0].scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', s=20, alpha=0.6)
axes[0].set_title('Teljes címkézett adat')

# Semi-supervised setup
axes[1].scatter(X_unlabeled[:, 0], X_unlabeled[:, 1], c='gray', s=20, alpha=0.4, label='Címkézetlen')
axes[1].scatter(X_labeled[y_labeled==0, 0], X_labeled[y_labeled==0, 1],
                c='blue', s=100, marker='s', label='Címkézett (0)')
axes[1].scatter(X_labeled[y_labeled==1, 0], X_labeled[y_labeled==1, 1],
                c='red', s=100, marker='s', label='Címkézett (1)')
axes[1].set_title(f'Semi-supervised: {n_labeled} címkézett, {len(X_unlabeled)} címkézetlen')
axes[1].legend()

plt.tight_layout()
plt.show()

## 2. Pseudo-labeling algoritmus

### Algoritmus

```
PseudoLabeling(L, U, θ):
    # L: címkézett, U: címkézetlen, θ: threshold
    
    model.fit(L)
    
    for iteration in range(max_iter):
        # Predikció címkézetlenekre
        probs = model.predict_proba(U)
        pseudo_labels = argmax(probs)
        confidence = max(probs)
        
        # Magas konfidenciájú példák
        high_conf = U[confidence > θ]
        pseudo = pseudo_labels[confidence > θ]
        
        # Újratanítás
        L_new = L ∪ (high_conf, pseudo)
        model.fit(L_new)
    
    return model
```

### Kulcslépések

| Lépés | Leírás |
|-------|--------|
| 1 | Tanítás címkézetten |
| 2 | Predikció címkézetlenekre |
| 3 | Konfidencia szűrés |
| 4 | Bővített tanítás |

In [ ]:
def pseudo_labeling(X_labeled, y_labeled, X_unlabeled, X_test, y_test,
                   threshold=0.95, max_iter=10, verbose=True):
    """
    Egyszerű pseudo-labeling implementáció.
    """
    # Másolatok
    X_l = X_labeled.copy()
    y_l = y_labeled.copy()
    X_u = X_unlabeled.copy()

    history = {'accuracy': [], 'n_pseudo': [], 'n_unlabeled': []}

    for iteration in range(max_iter):
        # Tanítás
        model = LogisticRegression(random_state=42)
        model.fit(X_l, y_l)

        # Kiértékelés
        test_acc = accuracy_score(y_test, model.predict(X_test))

        if len(X_u) == 0:
            if verbose:
                print(f"Iteráció {iteration}: Nincs több címkézetlen adat")
            break

        # Predikció címkézetlenekre
        probs = model.predict_proba(X_u)
        max_probs = np.max(probs, axis=1)
        pseudo_labels = model.predict(X_u)

        # Magas konfidenciájú példák
        high_conf_mask = max_probs >= threshold
        n_pseudo = np.sum(high_conf_mask)

        history['accuracy'].append(test_acc)
        history['n_pseudo'].append(n_pseudo)
        history['n_unlabeled'].append(len(X_u))

        if verbose:
            print(f"Iteráció {iteration}: Acc={test_acc:.4f}, "
                  f"Pseudo={n_pseudo}, Marad={len(X_u)-n_pseudo}")

        if n_pseudo == 0:
            break

        # Hozzáadás címkézetthez
        X_l = np.vstack([X_l, X_u[high_conf_mask]])
        y_l = np.concatenate([y_l, pseudo_labels[high_conf_mask]])

        # Eltávolítás címkézetlenből
        X_u = X_u[~high_conf_mask]

    return model, history

# Futtatás
print("Pseudo-labeling folyamat:")
print("="*50)
model_pl, history = pseudo_labeling(X_labeled, y_labeled, X_unlabeled,
                                    X_test, y_test, threshold=0.9)

In [ ]:
# Összehasonlítás: csak címkézett vs pseudo-labeling

# Csak címkézett
model_supervised = LogisticRegression(random_state=42)
model_supervised.fit(X_labeled, y_labeled)
acc_supervised = accuracy_score(y_test, model_supervised.predict(X_test))

# Pseudo-labeling
acc_pseudo = accuracy_score(y_test, model_pl.predict(X_test))

# Teljes címkézett (upper bound)
model_full = LogisticRegression(random_state=42)
model_full.fit(X_train, y_train)
acc_full = accuracy_score(y_test, model_full.predict(X_test))

print(f"Csak {n_labeled} címkézett: {acc_supervised:.4f}")
print(f"Pseudo-labeling: {acc_pseudo:.4f}")
print(f"Teljes címkézett (upper bound): {acc_full:.4f}")
print(f"\nJavulás: +{(acc_pseudo - acc_supervised)*100:.1f}%")

## 3. Confidence threshold

### Threshold hatása

| Threshold | Előny | Hátrány |
|-----------|-------|--------|
| Magas (0.99) | Kevés hibás címke | Lassú tanulás |
| Alacsony (0.7) | Gyors bővítés | Hibás címkék |

### Optimális választás

- Kezdetben magas threshold
- Fokozatosan csökkentés
- Validációs halmazon hangolás

In [ ]:
# Threshold hatásának vizsgálata

thresholds = [0.6, 0.7, 0.8, 0.9, 0.95, 0.99]
results = []

for thresh in thresholds:
    model, hist = pseudo_labeling(X_labeled, y_labeled, X_unlabeled,
                                  X_test, y_test, threshold=thresh, verbose=False)
    acc = accuracy_score(y_test, model.predict(X_test))
    results.append({
        'threshold': thresh,
        'accuracy': acc,
        'iterations': len(hist['accuracy'])
    })

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy vs threshold
accs = [r['accuracy'] for r in results]
axes[0].plot(thresholds, accs, 'bo-', linewidth=2, markersize=10)
axes[0].axhline(y=acc_supervised, color='r', linestyle='--', label=f'Supervised: {acc_supervised:.3f}')
axes[0].axhline(y=acc_full, color='g', linestyle='--', label=f'Full data: {acc_full:.3f}')
axes[0].set_xlabel('Confidence threshold')
axes[0].set_ylabel('Test accuracy')
axes[0].set_title('Threshold hatása pontosságra')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Iterations vs threshold
iters = [r['iterations'] for r in results]
axes[1].bar(range(len(thresholds)), iters, color='steelblue')
axes[1].set_xticks(range(len(thresholds)))
axes[1].set_xticklabels([str(t) for t in thresholds])
axes[1].set_xlabel('Confidence threshold')
axes[1].set_ylabel('Iterációk száma')
axes[1].set_title('Konvergencia sebesség')

plt.tight_layout()
plt.show()

In [ ]:
# Konfidencia eloszlás vizualizáció

model_init = LogisticRegression(random_state=42)
model_init.fit(X_labeled, y_labeled)

probs = model_init.predict_proba(X_unlabeled)
max_probs = np.max(probs, axis=1)
predictions = model_init.predict(X_unlabeled)
correct = predictions == y_unlabeled_true

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Konfidencia eloszlás
axes[0].hist(max_probs[correct], bins=20, alpha=0.7, label='Helyes', color='green')
axes[0].hist(max_probs[~correct], bins=20, alpha=0.7, label='Hibás', color='red')
axes[0].axvline(x=0.9, color='black', linestyle='--', label='θ=0.9')
axes[0].set_xlabel('Konfidencia')
axes[0].set_ylabel('Gyakoriság')
axes[0].set_title('Konfidencia eloszlás')
axes[0].legend()

# Pontosság vs konfidencia
conf_bins = np.linspace(0.5, 1.0, 11)
bin_accs = []

for i in range(len(conf_bins)-1):
    mask = (max_probs >= conf_bins[i]) & (max_probs < conf_bins[i+1])
    if np.sum(mask) > 0:
        bin_accs.append(np.mean(correct[mask]))
    else:
        bin_accs.append(np.nan)

axes[1].bar(range(len(bin_accs)), bin_accs, color='steelblue')
axes[1].set_xticks(range(len(bin_accs)))
axes[1].set_xticklabels([f'{conf_bins[i]:.1f}-{conf_bins[i+1]:.1f}'
                         for i in range(len(conf_bins)-1)], rotation=45)
axes[1].set_xlabel('Konfidencia tartomány')
axes[1].set_ylabel('Pontosság')
axes[1].set_title('Pontosság konfidencia szerint')

plt.tight_layout()
plt.show()

## 4. Self-training

### Sklearn SelfTrainingClassifier

A scikit-learn beépített implementációja:

```python
SelfTrainingClassifier(
    base_estimator,  # Alap osztályozó
    threshold=0.75,  # Konfidencia küszöb
    criterion='threshold',  # 'threshold' vagy 'k_best'
    k_best=10,  # Top-k pseudo-label
    max_iter=10  # Max iteráció
)
```

In [ ]:
# Sklearn SelfTrainingClassifier

# Semi-supervised label setup: -1 jelzi a címkézetlent
y_train_semi = np.full(len(X_train), -1)  # Minden címkézetlen
y_train_semi[labeled_idx] = y_train[labeled_idx]  # Csak néhány címkézett

# Különböző kritériumok
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Threshold kritérium
self_train_thresh = SelfTrainingClassifier(
    base_estimator=LogisticRegression(random_state=42),
    threshold=0.9,
    criterion='threshold',
    max_iter=10
)
self_train_thresh.fit(X_train, y_train_semi)
acc_thresh = accuracy_score(y_test, self_train_thresh.predict(X_test))

axes[0].scatter(X_test[:, 0], X_test[:, 1],
                c=self_train_thresh.predict(X_test), cmap='coolwarm', s=20)
axes[0].set_title(f'Threshold (θ=0.9)\nAcc: {acc_thresh:.4f}')

# 2. K-best kritérium
self_train_kbest = SelfTrainingClassifier(
    base_estimator=LogisticRegression(random_state=42),
    criterion='k_best',
    k_best=20,
    max_iter=10
)
self_train_kbest.fit(X_train, y_train_semi)
acc_kbest = accuracy_score(y_test, self_train_kbest.predict(X_test))

axes[1].scatter(X_test[:, 0], X_test[:, 1],
                c=self_train_kbest.predict(X_test), cmap='coolwarm', s=20)
axes[1].set_title(f'K-best (k=20)\nAcc: {acc_kbest:.4f}')

# 3. MLP base estimator
self_train_mlp = SelfTrainingClassifier(
    base_estimator=MLPClassifier(hidden_layer_sizes=(50,), max_iter=500, random_state=42),
    threshold=0.9,
    max_iter=10
)
self_train_mlp.fit(X_train, y_train_semi)
acc_mlp = accuracy_score(y_test, self_train_mlp.predict(X_test))

axes[2].scatter(X_test[:, 0], X_test[:, 1],
                c=self_train_mlp.predict(X_test), cmap='coolwarm', s=20)
axes[2].set_title(f'MLP base\nAcc: {acc_mlp:.4f}')

plt.suptitle('SelfTrainingClassifier variációk', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Gyakorlati alkalmazás

### Mikor használd

| Szituáció | Pseudo-labeling |
|-----------|----------------|
| Kevés címkézett | ✓ Jó választás |
| Sok címkézetlen | ✓ Ideális |
| Zajos adat | ⚠ Óvatosan |
| Osztályegyensúlytalanság | ⚠ Óvatosan |

In [ ]:
# Nagyobb példa: több osztály

X_multi, y_multi = make_classification(
    n_samples=1000, n_features=20, n_informative=15,
    n_redundant=5, n_classes=5, n_clusters_per_class=1,
    random_state=42
)

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi, y_multi, test_size=0.3, random_state=42
)

# Különböző címkézett arányok
label_ratios = [0.01, 0.02, 0.05, 0.1, 0.2, 0.5]
results_sup = []
results_semi = []

for ratio in label_ratios:
    n_labeled = int(len(X_train_m) * ratio)
    labeled_idx = np.random.choice(len(X_train_m), n_labeled, replace=False)

    # Supervised
    model_sup = LogisticRegression(random_state=42, max_iter=1000)
    model_sup.fit(X_train_m[labeled_idx], y_train_m[labeled_idx])
    acc_sup = accuracy_score(y_test_m, model_sup.predict(X_test_m))
    results_sup.append(acc_sup)

    # Semi-supervised
    y_semi = np.full(len(X_train_m), -1)
    y_semi[labeled_idx] = y_train_m[labeled_idx]

    model_semi = SelfTrainingClassifier(
        base_estimator=LogisticRegression(random_state=42, max_iter=1000),
        threshold=0.9,
        max_iter=20
    )
    model_semi.fit(X_train_m, y_semi)
    acc_semi = accuracy_score(y_test_m, model_semi.predict(X_test_m))
    results_semi.append(acc_semi)

# Vizualizáció
plt.figure(figsize=(10, 6))
plt.plot(label_ratios, results_sup, 'ro-', linewidth=2, markersize=10, label='Supervised')
plt.plot(label_ratios, results_semi, 'bs-', linewidth=2, markersize=10, label='Semi-supervised')
plt.xlabel('Címkézett adat aránya')
plt.ylabel('Test accuracy')
plt.title('Supervised vs Semi-supervised (5 osztály)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xscale('log')
plt.show()

# Javulás
print("\nJavulás a semi-supervised tanulással:")
for ratio, sup, semi in zip(label_ratios, results_sup, results_semi):
    improvement = (semi - sup) * 100
    print(f"  {ratio*100:5.1f}% címkézett: {sup:.3f} → {semi:.3f} (+{improvement:.1f}%)")

## Összefoglalás

### Pseudo-labeling:

| Tulajdonság | Leírás |
|-------------|--------|
| Cél | Címkézetlen adat használata |
| Módszer | Magas konfidenciájú predikciók |
| Iteratív | Fokozatos bővítés |

### Kritikus paraméter:

| Threshold | Hatás |
|-----------|-------|
| Magas | Kevés de pontos pseudo-label |
| Alacsony | Sok de zajos pseudo-label |

### Mikor hatékony:

- Kevés címkézett + sok címkézetlen
- Jó induló modell
- Smoothness feltevés teljesül